# 1.8 Backtracking: Implementation Snippet

In [37]:
# Variables and their domains
variables = ["A", "B", "C", "D"]

domains = {
    "A": ["Red", "Green", "Blue"],
    "B": ["Red", "Green", "Blue"],
    "C": ["Red", "Green", "Blue"],
    "D": ["Red", "Green", "Blue"]
}

# Constraints: pairs of variables that must be different
neighbors = [
    ("A", "B"),
    ("A", "C"),
    ("B", "C"),
    ("B", "D")
]

csp.constraints = {
    "A": ["B", "C"],
    "B": ["A", "C", "D"],
    "C": ["A", "B"],
    "D": ["B"]
}

csp.domains = {
    "A": ["Red", "Green", "Blue"],
    "B": ["Red", "Green", "Blue"],
    "C": ["Red", "Green", "Blue"],
    "D": ["Red", "Green", "Blue"]
}

In [38]:
def is_consistent(variable, value, assignment):
    # Check each constraint (pair of neighbors)
    for pair in neighbors:
        var1 = pair[0]
        var2 = pair[1]
        
        # Case 1: our variable is var1
        if variable == var1:
            # Check if var2 is already assigned the same value
            if var2 in assignment:
                if assignment[var2] == value:
                    return False
        
        # Case 2: our variable is var2
        if variable == var2:
            # Check if var1 is already assigned the same value
            if var1 in assignment:
                if assignment[var1] == value:
                    return False
    
    # No conflict found
    return True

In [39]:
def backtrack(assignment, csp):
    # If all variables are assigned, we found a solution
    if len(assignment) == len(csp.variables):
        return assignment
    
    # Heuristics: MRV and Degree Heuristics
    var = select_unassigned_variable_MRV_Degree(assignment, csp)
    
    # Heuristics: Least Constraining Value
    for value in order_domain_values_LCV(var, assignment, csp):
        if is_consistent(var, value, assignment, csp):
            assignment[var] = value
            
            # Optional: apply inference like AC-3 here (Forward Checking)
            inferences = apply_constraint_propagation(var, value, csp)
            if inferences != FAILURE:
                csp.add_inferences(inferences)
            
            result = backtrack(assignment, csp)
            if result is not None:
                return result
            
            csp.remove_inferences(inferences)
            del assignment[var]  # Undo assignment
    
    return None  # No solution found on this branch

### 1. select_unassigned_variable_MRV_Degree: Chooses variables intelligently to fail early and save time.

In [40]:
def select_unassigned_variable_MRV_Degree(assignment, csp):
    unassigned = []

    # to check which variables are not assigned
    for v in csp['variables']:
        if v not in assignment:
            unassigned.append(v) 

    best = None # to storee best pick
    best_remaining = float('inf') # totrack the small domains
    best_degree = -1 

    # loop through the unasiigned variables
    for var in unassigned:
        remaining = len(csp['domains'][var]) # to count the possible remianing values
        degree = 0 

        # to find the degree of the variaable
        for (v1, v2) in csp['constraints']:
            if var in (v1, v2):
                if v1 not in assignment and v2 not in assignment:
                    degree += 1

        # assigning values
        if remaining < best_remaining or (remaining == best_remaining and degree > best_degree):
            best = var
            best_remaining = remaining
            best_degree = degree

    return best

### 2. order_domain_values_LCV: Chooses values that leave the most options open for neighbors.

In [41]:
def order_domain_values_LCV(var, assignment, csp):
    
    # this will count the invalid neighbours for each choosen value
    def count_conflicts(value, var, assignment, csp):
        conflicts = 0
        
        # Check each constraint involving var
        for neighbor in csp.constraints.get(var, []):
            if neighbor not in assignment: 
                for neighbor_value in csp.domains[neighbor]:
                    if not csp.is_consistent_pair(var, value, neighbor, neighbor_value):
                        conflicts += 1
        return conflicts
    
    # variables having fewest conflicts will be first 
    values = list(csp.domains[var])
    
    def get_conflict_count_for_value(val):
        return count_conflicts(val, var, assignment, csp)
    
    # Then use it
    values.sort(key=get_conflict_count_for_value)
    return values

### 3. apply_constraint_propagation: Performing inference, like checking arc consistency, dynamically as assignments are made (an approach called Maintaining Arc Consistency or MAC).


# NOTE:
### Codes below are from the othere sections manual

# 1.4 Coding the Map Coloring CSP

### 1.4.1 Step 1: Define the CSP

In [73]:
# Variables and their domains
variables = ["A", "B", "C", "D"]

domains = {
    "A": ["Red", "Green", "Blue"],
    "B": ["Red", "Green", "Blue"],
    "C": ["Red", "Green", "Blue"],
    "D": ["Red", "Green", "Blue"]
}

# Constraints: pairs of variables that must be different
neighbors = [
    ("A", "B"),
    ("A", "C"),
    ("B", "C"),
    ("B", "D")
]

In [74]:
print(variables)  # [’A’, ’B’, ’C’, ’D’]
print(domains["A"]) # [’Red’, ’Green’, ’Blue’]
print(neighbors[0]) # (’A’, ’B’)

['A', 'B', 'C', 'D']
['Red', 'Green', 'Blue']
('A', 'B')


### 1.4.2 Step 2: Check if an Assignment is Consistent

In [75]:
def is_consistent(variable, value, assignment):
    # Check each constraint (pair of neighbors)
    for pair in neighbors:
        var1 = pair[0]
        var2 = pair[1]
        
        # Case 1: our variable is var1
        if variable == var1:
            # Check if var2 is already assigned the same value
            if var2 in assignment:
                if assignment[var2] == value:
                    return False
        
        # Case 2: our variable is var2
        if variable == var2:
            # Check if var1 is already assigned the same value
            if var1 in assignment:
                if assignment[var1] == value:
                    return False
    
    # No conflict found
    return True

In [76]:
assignment = {"A": "Red"}
print(is_consistent("B", "Red", assignment)) # False (A and B are neighbors)
print(is_consistent("B", "Green", assignment)) # True
print(is_consistent("D", "Red", assignment)) # True (D is not a neighbor of A)

False
True
True


### 1.4.3 Step 3: Backtracking Search

In [77]:
def backtrack(assignment):
    # If all variables are assigned, we found a solution!
    if len(assignment) == len(variables):
        return assignment
    
    # Pick the next unassigned variable
    var = None
    for v in variables:
        if v not in assignment:
            var = v
            break
    
    # Try each value in the domain
    for value in domains[var]:
        # Check if this value is safe to assign
        if is_consistent(var, value, assignment):
            # Assign the value
            assignment[var] = value
            
            # Continue to the next variable
            result = backtrack(assignment)
            if result is not None:
                return result
            
            # If it didn't work, undo (backtrack)
            del assignment[var]
    
    # No value worked for this variable
    return None

In [78]:
solution=backtrack({})
print(solution)
#You shouldget something like: {'A':'Red','B':'Green','C':'Blue','D':'Red'}
#Verify: IsA=B? IsA=C? IsB=C? IsB=D? Allyes? Then it’s correct!

{'A': 'Red', 'B': 'Green', 'C': 'Blue', 'D': 'Red'}


In [79]:
def backtrack_verbose(assignment):
    if len(assignment) == len(variables):
        return assignment
    
    var = None
    for v in variables:
        if v not in assignment:
            var = v
            break
    
    for value in domains[var]:
        if is_consistent(var, value, assignment):
            assignment[var] = value
            print("  Assign:", var, "=", value, " | Current:", assignment)
            
            result = backtrack_verbose(assignment)
            if result is not None:
                return result
            
            # Undo
            del assignment[var]
            print("  Backtrack: undo", var, "=", value)
    
    return None

print("Solving map coloring:")
solution = backtrack_verbose({})
print("Solution:", solution)

Solving map coloring:
  Assign: A = Red  | Current: {'A': 'Red'}
  Assign: B = Green  | Current: {'A': 'Red', 'B': 'Green'}
  Assign: C = Blue  | Current: {'A': 'Red', 'B': 'Green', 'C': 'Blue'}
  Assign: D = Red  | Current: {'A': 'Red', 'B': 'Green', 'C': 'Blue', 'D': 'Red'}
Solution: {'A': 'Red', 'B': 'Green', 'C': 'Blue', 'D': 'Red'}


# Questions:
# 1.
### At which variable does the algorithm first backtrack?
# ANS:
### The algorithm will first backtrack at B when we assign value to A 
### we cannot assign the same value to B so we backtrack at b
###
# 2.
### How many times does the algorithm backtrack in total?
# ANS:
### It will backtrack 3 times, 1 at b and 2 times at c
###
# 3.
### Does the algorithm try all possible combinations, or does it skip some? Why?
# ANS:
### It will assign values to the variables and 
### when all variables were assigned values then it will skip the remaining possibilities

# 1.6 Problem 2: N-Queens

### 1.6.1 Step 1: Define the N-Queens CSP

In [80]:
N = 12  # Try 4-Queens first, then increase to 8

# Variables: one per column (column 0, 1, 2, ...)
variables = []
for i in range(N):
    variables.append(i)

# Domains: each queen can go in any row (0 to N-1)
domains = {}
for col in variables:
    rows = []
    for r in range(N):
        rows.append(r)
    domains[col] = rows

### 1.6.2 Step2: Constraint Check for Queens

In [81]:
def is_consistent(col, row, assignment):
    # Check against every already-placed queen
    for other_col in assignment:
        other_row = assignment[other_col]
        
        # Same row?
        if row == other_row:
            return False
        
        # Same diagonal?
        row_diff = abs(row - other_row)
        col_diff = abs(col - other_col)
        if row_diff == col_diff:
            return False
    
    return True

In [82]:
assignment={0:1} #Queenincolumn0atrow1
print(is_consistent(1,1,assignment)) #False(samerow)
print(is_consistent(1,2,assignment)) #False(samediagonal)
print(is_consistent(1,3,assignment))

False
False
True


### 1.6.3 Step3: Solve with Backtracking

In [83]:
def backtrack(assignment):
    if len(assignment) == len(variables):
        return assignment
    
    # Pick the next unassigned column
    col = None
    for v in variables:
        if v not in assignment:
            col = v
            break
    
    # Try each row
    for row in domains[col]:
        if is_consistent(col, row, assignment):
            assignment[col] = row
            
            result = backtrack(assignment)
            if result is not None:
                return result
            
            del assignment[col]
    
    return None

solution = backtrack({})
print("Solution:", solution)

Solution: {0: 0, 1: 2, 2: 4, 3: 7, 4: 9, 5: 11, 6: 5, 7: 10, 8: 1, 9: 6, 10: 8, 11: 3}


# Checkpoint # 06:

In [84]:
solution = backtrack({})
print("Solution:", solution)

Solution: {0: 0, 1: 2, 2: 4, 3: 7, 4: 9, 5: 11, 6: 5, 7: 10, 8: 1, 9: 6, 10: 8, 11: 3}


### no it does not share a row column or diagonal

## 1.6.4 Step 4: Print the Board

In [85]:
def print_board(solution, n):
    for row in range(n):
        line = ""
        for col in range(n):
            if solution[col] == row:
                line = line + " Q"
            else:
                line = line + " ."
        print(line)

print_board(solution, N)

 Q . . . . . . . . . . .
 . . . . . . . . Q . . .
 . Q . . . . . . . . . .
 . . . . . . . . . . . Q
 . . Q . . . . . . . . .
 . . . . . . Q . . . . .
 . . . . . . . . . Q . .
 . . . Q . . . . . . . .
 . . . . . . . . . . Q .
 . . . . Q . . . . . . .
 . . . . . . . Q . . . .
 . . . . . Q . . . . . .


# 1.7 Experiment & Explore

## 1.7.1 Experiment 1: Increase N

### Question:
#### What happens as N increases? 
### ANS: 
#### time will increase
###
#### Why does it take longer?
### ANS:
#### Because of back tracking

#
# 1.7.2 Experiment 2: Change the Map

# Question: 
#### 1. Add a 5th region E that is a neighbor of both C and D. Can it still be solved with 3 colors?
# ANS:
#### Yes it can stil be solved, a=red, b=green, c=blue, d=red, e=green
#
# Question: 
#### 2. Make all 5 regions neighbors of each other. How many colors do you need now?
# ANS:
#### Then we will need 5 colors

#
# 1.8 Challenge: Sudoku Solver

In [86]:
board = [
    [5, 3, 0, 0, 7, 0, 0, 0, 0],
    [6, 0, 0, 1, 9, 5, 0, 0, 0],
    [0, 9, 8, 0, 0, 0, 0, 6, 0],
    [8, 0, 0, 0, 6, 0, 0, 0, 3],
    [4, 0, 0, 8, 0, 3, 0, 0, 1],
    [7, 0, 0, 0, 2, 0, 0, 0, 6],
    [0, 6, 0, 0, 0, 0, 2, 8, 0],
    [0, 0, 0, 4, 1, 9, 0, 0, 5],
    [0, 0, 0, 0, 8, 0, 0, 7, 9]
]

# finding the first index containin 0
def find_empty(board): 
    for row in range(9):
        for col in range(9):
            if board[row][col] == 0:
                return (row, col)
    return None

# this will check if the number is valid or not
def is_valid(board, row, col, num):
    for c in range(9):
        if board[row][c] == num: # if number already exist at this index then invalid
            return False
    
    for r in range(9):
        if board[r][col] == num:  # if number already exist at this index then invalid
            return False

    # this will find the starting positions of the box
    box_row = (row // 3) * 3
    box_col = (col // 3) * 3

    # if the number even exist in the box then it will also be invalid
    for r in range(box_row, box_row + 3):
        for c in range(box_col, box_col + 3):
            if board[r][c] == num:
                return False
    
    return True # if none of the above conditions satisfies then the number is valid

def solve(board):
    empty = find_empty(board)
    if not empty:
        return True # if no empty index found then puzzle solved
    
    row, col = empty # this is used to get position if an empty index is found

    # now we will try numbers from 1 to 9 
    for num in range(1, 10):
        if is_valid(board, row, col, num): # if the number is valid then assign that index the number
            board[row][col] = num
            
            if solve(board): # check if puzzle is solved completly or not
                return True
            
            board[row][col] = 0  # for backtracking 
    
    return False #if conditions not satisfied

# printing the board now
def print_board(board):
    for row in range(9):
        line = ""
        for col in range(9):
            line = line + str(board[row][col]) + " "
        print(line)

# Solve and print
if solve(board):
    print("Solved!")
    print_board(board)
else:
    print("No solution exists.")

Solved!
5 3 4 6 7 8 9 1 2 
6 7 2 1 9 5 3 4 8 
1 9 8 3 4 2 5 6 7 
8 5 9 7 6 1 4 2 3 
4 2 6 8 5 3 7 9 1 
7 1 3 9 2 4 8 5 6 
9 6 1 5 3 7 2 8 4 
2 8 7 4 1 9 6 3 5 
3 4 5 2 8 6 1 7 9 


# 1.9 Reflection

## Question 1:
#### In your own words, explain what “backtracking” means and why it is more efficient than trying every possible combination.
####
## Ans:
#### Backtracking means we try solutions step by step and if any constraint does not satisfies then
#### we will replace that option. it is efficient because it removes the variable that
#### does not satisfies the constraints and it is faster to check weather the solution exist or not
#
## Question 2:
#### For the map coloring problem, what is the minimum number of colors needed if every region is a neighbor of every other region? Why?
####
## ANS:
#### if every one is neighbour of each other then total number of variables = colors needed
#
## Question 3:
#### How does representing a problem as a CSP help us solve it? What are the advantages over writing a custom algorithm for each problem?
####
## ANS:
#### CSP helps us to reduce the values for the variables where they create issues in the constraint
#### its advantage is it removes all the values for the variables where
#### the solution combination is not possible and the remaining values satisfies all the combinations.
#
## Question 4:
#### What would happen to the N-Queens solver if we did not check constraints (i.e., we
#### just placed queens anywhere)? How many combinations would we need to check for N =8?
####
## ANS:
#### if we dont use constraints then there will be 8 power 8 possibilities, which makes program very slow